In [ ]:
!pip install datasets transformers accelerate peft evaluate matplotlib -q

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
dataset = load_dataset("dbpedia_14")

print(dataset['train'].column_names)
print(dataset['train'][0])

In [ ]:
# Hyperparameters
model_name = "roberta-base"
learning_rate = 2e-5
batch_size = 16
num_train_epochs = 10
weight_decay = 0.01
max_seq_length = 128
lora_r = 8
lora_alpha = 16
lora_dropout = 0.05

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(example):
    # Use 'content' column from DBpedia
    text = example['content']
    label = example['label']  # 0-13

    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=256
    )
    enc['labels'] = label
    return enc

tokenized_dataset = dataset.map(tokenize_fn, batched=True)

# Shuffle train split
tokenized_dataset["train"] = tokenized_dataset["train"].shuffle(seed=42)
#tokenized_dataset["validation"] = tokenized_dataset["validation"].shuffle(seed=42)
#tokenized_dataset["test"] = tokenized_dataset["test"].shuffle(seed=42)

for i in range(10):
    print(tokenized_dataset['train'][i]['labels'], tokenized_dataset['train'][i]['content'][:100])

In [ ]:
num_labels = 14

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

In [ ]:
# Lora Adapter
for name, module in base_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(name)

# Typical target_modules in RobertaAttention
target_modules = ["query", "key", "value"]

lora_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(base_model, lora_config)

# Move model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

In [ ]:
# Matrix
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": acc["accuracy"]}

In [ ]:
training_args = TrainingArguments(
    output_dir="./dbpedia",
    eval_strategy="steps",
    eval_steps=100,                  
    save_strategy="steps",
    save_steps=100,
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    weight_decay=weight_decay,
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="loss", # for early stopping
    greater_is_better=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(5000)),  # smaller subset for demo
    eval_dataset=tokenized_dataset["test"].select(range(1000)),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
train_result = trainer.train()
metrics = train_result.metrics
print("Training metrics:", metrics)

# Extract training and validation loss from log_history
log_history = trainer.state.log_history
train_loss = [x["loss"] for x in log_history if "loss" in x]
eval_loss = [x["eval_loss"] for x in log_history if "eval_loss" in x]

# Get the corresponding steps
train_steps = [x["step"] for x in log_history if "loss" in x]
eval_steps = [x["step"] for x in log_history if "eval_loss" in x]

# Plot both
plt.figure(figsize=(10,6))
plt.plot(train_steps, train_loss, label="Training Loss", marker='o')
plt.plot(eval_steps, eval_loss, label="Validation Loss", marker='x')
plt.title("Training & Validation Loss Over Steps")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation metrics:", eval_results)

In [ ]:
adapter_save_path = "./dbpedia_lora_adapter"
model.save_pretrained(adapter_save_path)
print(f"LoRA adapter saved to {adapter_save_path}")

In [ ]:
# Personal Test
dbpedia_labels = [
    "Company",
    "EducationalInstitution",
    "Artist",
    "Athlete",
    "OfficeHolder",
    "MeanOfTransportation",
    "Building",
    "NaturalPlace",
    "Village",
    "Animal",
    "Plant",
    "Album",
    "Film",
    "WrittenWork"
]

def predict_text(text):
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_seq_length
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_idx = torch.argmax(logits, dim=-1).item()
    return dbpedia_labels[pred_idx]

# Example prompts for all 14 DBpedia classes
examples = [
    "Apple Inc. is a multinational technology company known for its iPhone and Mac computers.",  # Company
    "Harvard University is a prestigious university located in Cambridge, Massachusetts.",     # EducationalInstitution
    "Beyoncé is a world-famous singer, songwriter, and performer.",                            # Artist
    "Usain Bolt is an Olympic sprinter from Jamaica who holds multiple world records.",        # Athlete
    "Barack Obama served as the 44th President of the United States.",                          # OfficeHolder
    "The Boeing 747 is a long-range wide-body airliner produced by Boeing.",                   # MeanOfTransportation
    "The Empire State Building is a 102-story skyscraper in New York City.",                   # Building
    "Mount Everest is the Earth's highest mountain, located in the Himalayas.",                # NaturalPlace
    "Greenwich is a town in London known for the Prime Meridian.",                             # Village
    "The African lion is a large carnivorous feline found in sub-Saharan Africa.",             # Animal
    "Rosa rubiginosa, also called sweet briar, is a species of rose native to Europe.",        # Plant
    "Abbey Road is a famous album released by the Beatles in 1969.",                           # Album
    "Inception is a 2010 science fiction film directed by Christopher Nolan.",                 # Film
    "Pride and Prejudice is a classic novel written by Jane Austen in 1813.",                  # WrittenWork
    "Singapore Mangement Univeristy is AWESOMMEEEEEEE."
]

# Run
for text in examples:
    pred_class = predict_text(text)
    print(f"Prompt: {text[:60]}... -> Predicted class: {pred_class}")